# Find geo targets for Personae

## Setup

In [2]:
# Import libraries
import re
import datetime
import time
import json
import pandas as pd
from google.cloud import storage

from google import genai
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

# Setup
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

# Global variables
PROJECT_ID = "ia-charles"
LOCATION = "us-central1"


## Call model

In [3]:
# Set up the Google GenAI client
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# Define a function to call Gemini
def call_gemini(prompt, system_instruction=None):
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=prompt,
        config=GenerateContentConfig(
            system_instruction=system_instruction,
            tools=[Tool(google_search=GoogleSearch())],
            temperature=0.2,
        ),
    )

    # Clean response text of anything before and after the JSON object
    response_text = response.text
    start_index = response_text.find("{")
    end_index = response_text.rfind("}") + 1
    if start_index == -1 or end_index == -1:
        raise ValueError("Invalid response format: JSON object not found.")
    json_response = response_text[start_index:end_index]

    # Convert to JSON
    json_response = json_response.replace("\\\"", "\"").replace("\\n", "\n")
    json_response = json_response.replace("\\", "")
    json_response = json_response.replace("\n", "")
    json_response = json_response.replace("}{", "},{")
    # Handle cases where the JSON object is not properly formatted
    if json_response.startswith("[") and json_response.endswith("]"):
        json_response = json_response[1:-1]
    elif json_response.startswith("{") and json_response.endswith("}"):
        pass
    else:
        raise ValueError("Invalid JSON format: JSON object not found.")

    # Return web search results
    source_documents = []
    for candidate in response.candidates:
        grounding = candidate.grounding_metadata
        if grounding and grounding.grounding_chunks:
            for chunk in grounding.grounding_chunks:
                retrieved_context = getattr(chunk, 'web', None)
                if retrieved_context:
                    source_info = {
                        'uri': getattr(retrieved_context, 'uri', None),
                        'title': getattr(retrieved_context, 'title', None)
                    }
                    source_documents.append(source_info)
    source_documents = list({v['uri']: v for v in source_documents if v['uri']}.values())

    # Return the JSON response and the grounded websites
    return json_response, source_documents

## Define system instructions

In [6]:
# General instruction template (use Python f-string placeholders for country-specific parts)
SYSTEM_INSTRUCTION_TEMPLATE = """
## Your Role
You are an expert Marketing Data Analyst specializing in geographic targeting inference for advertising campaigns within **{country_name}**. You are equipped with an **advanced web Search tool** which you must actively use to gather real-time information for your analysis.

## Your Core Task
Your primary task is to analyze a user-provided persona definition (including descriptive proxies and potential source ideas) and leverage your web research tool to identify **a comprehensive list** of {region_unit_plural} where this persona shows **significant signals** of being geographically concentrated or 'over-indexed' compared to a general {country_name} baseline. The goal is to generate a reasonably inclusive list based on detected signals, **not necessarily limited to only the absolute highest concentrations or top-percentile areas.** Do not aim for a round or pre‑set number of {region_unit_plural}. The list length should emerge naturally from the strength of your evidence.

## Key Capabilities
* You can understand detailed persona descriptions and associated data proxies.
* You can formulate effective, geographically focused search queries (for {country_name}) based on behavioral, demographic, and psychographic indicators.
* You **must** utilize your integrated web Search tool to execute these queries and gather relevant information from online sources (news articles, statistical summaries, directory listings, official website snippets if found, etc.).
* You can synthesize fragmented information retrieved from web searches to identify geographic areas where multiple persona indicators converge.
* You can infer likely areas of concentration and make a best effort to map these findings to specific {region_unit_plural}.

## Expected User Input
The user will provide their request in a **User Prompt**. This prompt will contain:
1.  **Persona Analysis Data:** A JSON object describing the target persona. This JSON object **must** include structures detailing:
    * `persona_name` (string)
    * `persona_definition` (string)
    * `data_proxies` (an array of objects, each containing `proxy_category` and `indicators` which are strings describing measurable characteristics or behaviors relevant to the persona).
    * `data_sources` (an array of objects detailing potential general source types - you should treat these primarily as **inspiration or hints** for formulating your web search queries, not as sources you can directly query unless your tool specifically allows interaction with a listed URL).
2.  **Implicit Target Geography:** The analysis scope is always **{country_name}**.

## Processing Instructions & Methodology
1.  **Analyze Persona & Proxies:** Carefully examine the user-provided JSON, focusing on the specific `indicators` within `data_proxies`. These are the core signals to investigate geographically using your web tool.
2.  **Develop Web Search Strategy:** Based on the `indicators` (and using `data_sources` for ideas), devise targeted search queries aimed at finding geographic evidence within {country_name}. **Explore broadly:** look for evidence in major hubs, secondary cities, specific {region_unit_plural}, or types of regions (e.g., affluent suburban areas, startup ecosystems, rural hubs) suggested by the persona.
3.  **Execute Web Research & Extract:** Actively use your web Search tool for each relevant query. Extract specific location mentions (cities, {region_unit_plural}, metropolitan areas, distinct communities) that show **notable correlation** with the persona proxies based on the content retrieved.
4.  **Synthesize & Infer Concentration (Broader Threshold):** Combine the evidence gathered across multiple searches. Identify geographic areas where web search results show **significant evidence** for **one or more key proxies**, OR **moderate convergence** across several proxies. Aim to include areas that show clear positive signals, even if they aren't the absolute peak.
5.  **Map to {region_unit_plural} (Best Effort & Broader Scope):** For **all promising cities, {region_unit_plural}, metropolitan areas, or distinct communities** identified through synthesis, determine the corresponding {region_unit_singular}(s). Include them all if multiple {region_unit_plural} show relevance based on your findings.
6.  **Justify:** Your justification for including each {region_unit_singular} **must** reference the proxies that were supported by the online findings and the information retrieved via your web Browse tool.
7.  **Acknowledge Limitations:** If web search yields sparse or conflicting data for certain areas or proxies, or if mapping is difficult, reflect this in the justification or confidence level. It is acceptable to include areas with "Medium" confidence if some positive signal was detected and aligns with the goal of a broader list.

## Output Requirements
* Avoid numerical padding. Return each {region_unit_singular} only if you can formulate a clear, proxy‑based justification and assign a confidence level. If no justification → exclude it.
* Your **entire response MUST be a single, valid JSON object.**
* **Strictly adhere** to this format. Do **not** include any introductory text, closing text, summaries outside the JSON, conversational remarks, or markdown code blocks (like ```json ... ```) around the JSON output.
* The JSON object must contain a single top-level key: `{output_key}`.
* The value of `{output_key}` must be an array of objects.
* Each object in the array represents a potentially over-indexed {region_unit_singular} and **must** have the following keys:
    * `{region_code_key}`: (String) The official code or number.
    * `location_context`: (String) The name of the {region_unit_singular} for reference (e.g., "{region_example}").
    * `justification`: (String) A brief explanation **in English** summarizing findings that support the over-indexing for this {region_unit_singular}, linking back to specific persona proxies.
    * `confidence_indication`: (Integer) A confidence score between 0 and 10 (0 is not confident at all, 10 is very confident) based on the strength, consistency, and convergence of evidence retrieved via the web search tool for this specific area.

**Example Output Structure:**
{example_output}
"""

# Example: France-specific variable
COUNTRY_SPECIFIC_FRANCE = {
    "country_name": "France",
    "region_unit": "département",
    "region_code_key": "region_code",
    "region_example": "Gironde",
    "output_key": "likely_overindexed_region",
    "example_output": '''{
    "likely_overindexed_region": [
        {
            "region_code": "33",
            "location_context": "Gironde",
            "justification": "High density of eco-conscious startups (Proxy A), reports of strong organic food retail growth (Proxy B), and elevated cycling infrastructure investment (Proxy C) based on local news and government reports.",
            "confidence_indication": 7
        }
        // ... more objects for other identified départements
    ]
}'''
}

COUNTRY_SPECIFIC_GERMANY = {
    "country_name": "Germany",
    "region_unit": "Landkreis",
    "region_code_key": "region_code",
    "region_example": "München",
    "output_key": "likely_overindexed_region",
    "example_output": '''{
    "likely_overindexed_region": [
        {
            "location_context": "München",
            "justification": "High density of tech startups (Proxy A), strong presence of automotive industry (Proxy B), and high disposable income (Proxy C) based on local news and economic reports.",
            "confidence_indication": 5
        }
        // ... more objects for other identified Landkreise
    ]
}'''
}

COUNTRY_SPECIFIC_JAPAN = {
    "country_name": "Japan",
    "region_unit_singular": "Prefecture",
    "region_unit_plural": "Prefectures",
    "region_code_key": "region_code",
    "region_example": "Okayama",
    "output_key": "likely_overindexed_regions",
    "example_output": '''{
    "likely_overindexed_region": [
        {
            "region_code": "33",
            "location_context": "Okayama",
            "justification": "High density of tech startups (Proxy A), strong presence of automotive industry (Proxy B), and high disposable income (Proxy C) based on local news and economic reports.",
            "confidence_indication": 5
        }
        // ... more objects for other identified Prefectures
    ]
}'''
}

COUNTRY_SPECIFIC_ITALY = {
    "country_name": "Italy",
    "region_unit_singular": "Province",
    "region_unit_plural": "Provinces",
    "region_code_key": "region_code",
    "region_example": "Trento",
    "output_key": "likely_overindexed_regions",
    "example_output": '''{
    "likely_overindexed_region": [
        {
            "region_code": "TN",
            "location_context": "Trento",
            "justification": "High density of tech startups (Proxy A), strong presence of automotive industry (Proxy B), and high disposable income (Proxy C) based on local news and economic reports.",
            "confidence_indication": 5
        }
        // ... more objects for other identified Province
        ]
}'''
}

COUNTRY_SPECIFIC_UK = {
    "country_name": "Great-Britain",
    "region_unit_singular": "County",
    "region_unit_plural": "Counties",
    "region_code_key": "region_code",
    "region_example": "Essex",
    "output_key": "likely_overindexed_regions",
    "example_output": '''{
    "likely_overindexed_region": [
        {
            "region_code": "GB-ESS",
            "location_context": "Essex",
            "justification": "High density of tech startups (Proxy A), strong presence of automotive industry (Proxy B), and high disposable income (Proxy C) based on local news and economic reports.",
            "confidence_indication": 5
        }
        // ... more objects for other identified Province
        ]
}'''
}

COUNTRY_SPECIFIC_ES = {
    "country_name": "Spain",
    "region_unit_singular": "Province",
    "region_unit_plural": "Provinces",
    "region_code_key": "region_code",
    "region_example": "Guadalajara",
    "output_key": "likely_overindexed_regions",
    "example_output": '''{
    "likely_overindexed_region": [
        {
            "region_code": "GU",
            "location_context": "Guadalajara",
            "justification": "High density of tech startups (Proxy A), strong presence of automotive industry (Proxy B), and high disposable income (Proxy C) based on local news and economic reports.",
            "confidence_indication": 5
        }
        // ... more objects for other identified Province
        ]
}'''
}

COUNTRY_SPECIFIC_US = {
    "country_name": "United States",
    "region_unit_singular": "State",
    "region_unit_plural": "States",
    "region_code_key": "region_code",
    "region_example": "Florida",
    "output_key": "likely_overindexed_regions",
    "example_output": '''{
    "likely_overindexed_region": [
        {
            "region_code": "TX",
            "location_context": "Texas",
            "justification": "High density of tech startups (Proxy A), strong presence of automotive industry (Proxy B), and high disposable income (Proxy C) based on local news and economic reports.",
            "confidence_indication": 5
        }
        // ... more objects for other identified states
        ]
}'''
}

# Usage:
#SYSTEM_INSTRUCTION = SYSTEM_INSTRUCTION_TEMPLATE.format(**COUNTRY_SPECIFIC_FRANCE)
#SYSTEM_INSTRUCTION = SYSTEM_INSTRUCTION_TEMPLATE.format(**COUNTRY_SPECIFIC_GERMANY)
#SYSTEM_INSTRUCTION = SYSTEM_INSTRUCTION_TEMPLATE.format(**COUNTRY_SPECIFIC_JAPAN)
#SYSTEM_INSTRUCTION = SYSTEM_INSTRUCTION_TEMPLATE.format(**COUNTRY_SPECIFIC_ITALY)
#SYSTEM_INSTRUCTION = SYSTEM_INSTRUCTION_TEMPLATE.format(**COUNTRY_SPECIFIC_UK)
#SYSTEM_INSTRUCTION = SYSTEM_INSTRUCTION_TEMPLATE.format(**COUNTRY_SPECIFIC_ES)
SYSTEM_INSTRUCTION = SYSTEM_INSTRUCTION_TEMPLATE.format(**COUNTRY_SPECIFIC_US)

print(SYSTEM_INSTRUCTION)



## Your Role
You are an expert Marketing Data Analyst specializing in geographic targeting inference for advertising campaigns within **United States**. You are equipped with an **advanced web Search tool** which you must actively use to gather real-time information for your analysis.

## Your Core Task
Your primary task is to analyze a user-provided persona definition (including descriptive proxies and potential source ideas) and leverage your web research tool to identify **a comprehensive list** of Statess where this persona shows **significant signals** of being geographically concentrated or 'over-indexed' compared to a general United States baseline. The goal is to generate a reasonably inclusive list based on detected signals, **not necessarily limited to only the absolute highest concentrations or top-percentile areas.** Do not aim for a round or pre‑set number of Statess. The list length should emerge naturally from the strength of your evidence.

## Key Capabilities
* You ca

## Call model for each Persona

In [ ]:
# Read existing file for all countries
merged_responses_step1 = pd.read_json('/content/persona_gemini_step1_every_country_20250512_175702.json')

# GCS connection
BUCKET_NAME = "iacharles"
client_gcs = storage.Client()
bucket = client_gcs.bucket(BUCKET_NAME)

responses_step2 = {}
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
gcs_path = f"geo/persona_gemini_step2_responses_{timestamp}.json"
blob = bucket.blob(gcs_path)

boundaries_from = 0
boundaries_to = 3000

for key, step1_response in list(merged_responses_step1.items())[boundaries_from:boundaries_to]:
    attempts = 0
    while attempts < 5:
        try:
            step2_prompt = f"""
                ## Persona name: {step1_response['persona_name']}

                ## Persona description: {step1_response['persona_description']}

                ## Data Proxies: {json.dumps(step1_response['data_proxies'])}

                ## Data Sources: {json.dumps(step1_response['data_sources'])}
            """
            json_response, source_documents = call_gemini(step2_prompt, SYSTEM_INSTRUCTION)

            responses_step2[key] = {
                "json_response": json_response,
                "source_documents": source_documents
            }
            print(f"Response received for {key}")

            # Upload the updated responses dict to GCS
            blob.upload_from_string(
                json.dumps(responses_step2, indent=2),
                content_type="application/json"
            )
            break  # success → exit retry loop

        except Exception as e:
            err_str = str(e)
            # Check for 429/resource‐exhausted in the exception message
            if "429" in err_str or "503" in err_str:
                attempts += 1
                if attempts < 5:
                    print(f"Rate‐limit hit for {key} (attempt {attempts}). Waiting 5 seconds before retry...")
                    time.sleep(5)
                    continue
                else:
                    print(f"Failed after 5 retries for {key}: {e}")
                    break
            else:
                # Non‐retryable error
                print(f"Error for key: {key}")
                print(f"Exception: {e}")
                break
    # end retry loop

print(f"\nAll processed responses up to index {boundaries_to} have been written to gs://{BUCKET_NAME}/{gcs_path}")


Response received for City Dwellers
Response received for Rural Inhabitants
Response received for High Income Earners
Response received for UHNW (Ultra-High Net Worth) Elites
Response received for Hybrid & Electric Vehicle Seekers
Response received for Business Decision-Makers
Response received for Fitness Enthusiasts
Response received for Tech Enthusiasts
Response received for Price-Conscious Shoppers
Response received for Beauty Lovers
Response received for Parents of Young Children
Response received for New Parents / Expectant Parents
Response received for Travelers


## Convert result to Dataframe

In [ ]:
# Read JSON file
response_file_path = 'geo/persona_gemini_step2_responses_20250619_081608.json'
blob   = bucket.blob(response_file_path)
raw_bytes = blob.download_as_string()
data_dict = json.loads(raw_bytes)


# Convert to dataframe
rows = []

for persona_name, content in data_dict.items():
    raw = content.get("json_response")

    # If it's still a JSON string, parse it. If it's already a dict, skip loading.
    if isinstance(raw, str):
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError as e:
            print(f"Skipping {persona_name}—invalid JSON: {e}")
            continue
    elif isinstance(raw, dict):
        parsed = raw
    else:
        print(f"Skipping {persona_name}—unexpected type for json_response: {type(raw)}")
        continue

    # Extract source documents
    source_docs = content.get("source_documents", [])
    source_websites = [doc.get("title") for doc in source_docs if isinstance(doc, dict) and "title" in doc]

    for region in parsed.get("likely_overindexed_region", []):
        rows.append({
            "persona_name": persona_name,
            "region_code": region.get("region_code"),
            "location_context": region.get("location_context"),
            "justification": region.get("justification"),
            "confidence_indication": region.get("confidence_indication"),
            "sources": source_websites
        })

df = pd.DataFrame(rows)
df

Skipping DE&I Advocates—invalid JSON: Expecting ',' delimiter: line 1 column 6975 (char 6974)


""


## Save dataframe as CSV file to GCS

In [ ]:
gcs_csv_path = "geo/japan_part1_20250606.csv"
blob = bucket.blob(gcs_csv_path)
csv_string = df.to_csv(index=False)
blob.upload_from_string(csv_string, content_type="text/csv")

## Analyze results

In [ ]:
# Import libraries
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sns.set()

# Load data
df_zipcodes = (
    pd.read_csv("data/persona_locations_20250429.csv")
    .rename(columns={"departement_code": "zip_code"})
)

# Basic statistics
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(data=df_zipcodes.groupby("persona_name").nunique()['zip_code'], binwidth=1, ax=ax[0])
sns.barplot(data=df_zipcodes.groupby("zip_code").count()['persona_name'].sort_values(ascending=False).head(20), orient='h', ax=ax[1])

ax[0].set_title("Number of locations per persona")
ax[0].set_xlabel("")

ax[1].set_title("Number of personas associated with each location")
ax[1].set_xlabel("")